# 🌞 Solar Power Output Forecasting
## Final Year B.E./B.Tech Project
**Dataset:** Hourly Solar PV Generation Data — Norway  
**Target Variable:** `AC_Power` (kW)  
**Models Compared:** Linear Regression · Naive Bayes · XGBoost · LSTM · Transformer (MHA + FFN) · ARIMA · SARIMA

---

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBRegressor

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Dense, LSTM, Dropout,
    MultiHeadAttention, LayerNormalization,
    GlobalAveragePooling1D, Conv1D, Add
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

import warnings
warnings.filterwarnings("ignore")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✅ All libraries imported successfully.")
print(f"TensorFlow version : {tf.__version__}")

## 2. Load Dataset
Load the raw CSV and display basic statistics to understand the data structure.

In [ ]:
df = pd.read_csv("your_new_dataset.csv")

print(f"Shape            : {df.shape}")
print(f"Columns          : {list(df.columns)}")
print("\nFirst 5 rows:")
df.head()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)
print("\nDescriptive statistics:")
df.describe()

## 3. Data Preprocessing
Steps performed:
1. Parse `Date` + `Time` → `DateTime` index  
2. Sort chronologically  
3. Drop identifier columns (`Country`, `Date`, `Time`)  
4. Reindex to a full continuous hourly range (fill timestamp gaps)  
5. Forward-fill → backward-fill missing values  
6. Clip negative irradiation & power readings (sensor noise)  
7. IQR-based outlier capping on `AC_Power`

In [ ]:
# --- 3a. Parse DateTime ---
df['DateTime'] = pd.to_datetime(
    df['Date'].astype(str) + ' ' +
    df['Time'].astype(str).str.zfill(2) + ':00:00'
)
df = df.sort_values('DateTime').reset_index(drop=True)

# --- 3b. Drop identifier columns ---
df.drop(columns=['Date', 'Time', 'Country'], inplace=True)
df.set_index('DateTime', inplace=True)

# --- 3c. Remove duplicate timestamps ---
df = df[~df.index.duplicated(keep='first')]

# --- 3d. Reindex to ensure full hourly continuity ---
full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='H')
df = df.reindex(full_range)

# --- 3e. Handle missing values ---
df = df.fillna(method='ffill').fillna(method='bfill')

# --- 3f. Remove negative irradiation (sensor noise) ---
if 'Irradiation' in df.columns:
    df['Irradiation'] = df['Irradiation'].clip(lower=0)

# --- 3g. Clip AC/DC power: no negative generation ---
for col in ['AC_Power', 'DC_Power']:
    if col in df.columns:
        df[col] = df[col].clip(lower=0)

# --- 3h. Outlier removal using IQR on AC_Power ---
Q1 = df['AC_Power'].quantile(0.25)
Q3 = df['AC_Power'].quantile(0.75)
IQR = Q3 - Q1
df['AC_Power'] = df['AC_Power'].clip(upper=Q3 + 3 * IQR)

print(f"Cleaned shape : {df.shape}")
df.head()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Exploratory Data Analysis — Solar PV Dataset", fontsize=14, fontweight='bold')

# AC Power over time
axes[0, 0].plot(df['AC_Power'], linewidth=0.5, color='royalblue', alpha=0.7)
axes[0, 0].set_title("AC Power over Time")
axes[0, 0].set_ylabel("AC Power (kW)")

# Distribution
axes[0, 1].hist(df['AC_Power'], bins=60, color='steelblue', edgecolor='k', linewidth=0.4)
axes[0, 1].set_title("AC Power Distribution")
axes[0, 1].set_xlabel("AC Power (kW)")

# Average hourly generation
hourly_avg = df.groupby(df.index.hour)['AC_Power'].mean()
axes[1, 0].bar(hourly_avg.index, hourly_avg.values, color='orange', edgecolor='k', linewidth=0.4)
axes[1, 0].set_title("Average AC Power by Hour of Day")
axes[1, 0].set_xlabel("Hour"); axes[1, 0].set_ylabel("Mean AC Power (kW)")

# Monthly average
monthly_avg = df.groupby(df.index.month)['AC_Power'].mean()
axes[1, 1].bar(monthly_avg.index, monthly_avg.values, color='seagreen', edgecolor='k', linewidth=0.4)
axes[1, 1].set_title("Average AC Power by Month")
axes[1, 1].set_xlabel("Month"); axes[1, 1].set_ylabel("Mean AC Power (kW)")
axes[1, 1].set_xticks(range(1, 13))
axes[1, 1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                             'Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)

plt.tight_layout()
plt.savefig("eda_plots.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title("Feature Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Engineering
Features created:
- **Cyclical encoding** of Hour, Month, Day-of-Year (sin/cos) to avoid discontinuity  
- **Physics-inspired** interaction: Irradiation × temperature derating factor  
- **Lag features** at 1, 2, 3, 6, 12, 24 hours  
- **Rolling statistics** (mean & std) at 3, 6, 12, 24-hour windows  
- **Exponential weighted mean** at span 6 and 24

In [ ]:
# Temporal features
df['Hour']       = df.index.hour
df['DayOfWeek']  = df.index.dayofweek
df['Month']      = df.index.month
df['DayOfYear']  = df.index.dayofyear
df['WeekOfYear'] = df.index.isocalendar().week.astype(int)

# Cyclical encoding
df['Hour_sin']  = np.sin(2 * np.pi * df['Hour'] / 24)
df['Hour_cos']  = np.cos(2 * np.pi * df['Hour'] / 24)
df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)
df['DOY_sin']   = np.sin(2 * np.pi * df['DayOfYear'] / 365)
df['DOY_cos']   = np.cos(2 * np.pi * df['DayOfYear'] / 365)

# Physics-inspired interaction
if 'Irradiation' in df.columns and 'Module_Temperature' in df.columns:
    df['Irrad_x_Temp'] = df['Irradiation'] * (1 - 0.004 * df['Module_Temperature'])

# Lag features
for lag in [1, 2, 3, 6, 12, 24]:
    df[f'AC_lag_{lag}'] = df['AC_Power'].shift(lag)

# Rolling statistics
for window in [3, 6, 12, 24]:
    df[f'roll_mean_{window}'] = df['AC_Power'].shift(1).rolling(window).mean()
    df[f'roll_std_{window}']  = df['AC_Power'].shift(1).rolling(window).std()

# Exponential weighted mean
df['ewm_6']  = df['AC_Power'].shift(1).ewm(span=6).mean()
df['ewm_24'] = df['AC_Power'].shift(1).ewm(span=24).mean()

df.dropna(inplace=True)

print(f"Feature count : {df.shape[1]} columns")
print(f"Final shape   : {df.shape}")
print(f"\nAll columns:\n{list(df.columns)}")

## 6. Train / Test Split & Scaling
- **80/20 chronological split** — no shuffle (preserves temporal order)  
- `MinMaxScaler` fit on training data only (no data leakage)  
- Separate `target_scaler` for inverse-transforming predictions to real kW values

In [ ]:
TARGET = 'AC_Power'

# Drop leakage columns — these are derived from AC_Power
DROP_COLS = [c for c in ['AC_Power', 'DC_Power', 'Daily_Yield', 'Total_Yield'] if c in df.columns]

X = df.drop(columns=DROP_COLS)
y = df[TARGET]

split = int(len(df) * 0.80)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}")
print(f"Feature count    : {X.shape[1]}")

# Scale
feat_scaler   = MinMaxScaler()
target_scaler = MinMaxScaler()

X_train_sc = feat_scaler.fit_transform(X_train)
X_test_sc  = feat_scaler.transform(X_test)

y_train_sc = target_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_sc  = target_scaler.transform(y_test.values.reshape(-1, 1)).ravel()

print("\n✅ Scaling complete.")

## 7. Sequence Builder (for LSTM & Transformer)
Creates sliding-window sequences of length `TIME_STEPS` (24 hours look-back).

In [ ]:
def build_sequences(X, y, steps=24):
    """Convert flat feature array → (samples, steps, features) sequences."""
    Xs, ys = [], []
    for i in range(len(X) - steps):
        Xs.append(X[i : i + steps])
        ys.append(y[i + steps])
    return np.array(Xs), np.array(ys)

TIME_STEPS = 24
N_FEATURES = X_train_sc.shape[1]

X_tr_seq, y_tr_seq = build_sequences(X_train_sc, y_train_sc, TIME_STEPS)
X_te_seq, y_te_seq = build_sequences(X_test_sc,  y_test_sc,  TIME_STEPS)

# Original-scale targets for sequence models (for metric calculation)
y_test_seq_orig = target_scaler.inverse_transform(y_te_seq.reshape(-1, 1)).ravel()

print(f"Train sequences shape : {X_tr_seq.shape}")
print(f"Test  sequences shape : {X_te_seq.shape}")

## 8. Evaluation Metric Helper

In [ ]:
results = []

def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    results.append([name, round(mae, 4), round(rmse, 4), round(r2, 4)])
    print(f"  {name:<28}  MAE={mae:8.4f}  RMSE={rmse:8.4f}  R²={r2:7.4f}")

# Shared Keras callbacks
early_stop = EarlyStopping(patience=10, restore_best_weights=True, verbose=0)
reduce_lr  = ReduceLROnPlateau(patience=5, factor=0.5, verbose=0)

## 9. Model 1 — Linear Regression (Baseline)
A simple linear model that maps scaled features directly to AC_Power.

In [ ]:
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
pred_lr = lr.predict(X_test_sc)

print("Linear Regression — Evaluation:")
evaluate("Linear Regression", y_test, pred_lr)

## 10. Model 2 — Gaussian Naive Bayes
Naive Bayes is a classifier, so we adapt it for regression by:
1. Quantising `AC_Power` into `N_BINS` bins  
2. Training the classifier on bin labels  
3. Mapping predicted bin → mean AC_Power of that bin

In [ ]:
N_BINS = 50
y_train_binned = pd.cut(y_train, bins=N_BINS, labels=False).fillna(0).astype(int)
bin_means = y_train.groupby(y_train_binned.values).mean()

nb = GaussianNB()
nb.fit(X_train_sc, y_train_binned)
pred_nb_bins = nb.predict(X_test_sc)
pred_nb = np.array([bin_means.get(b, bin_means.mean()) for b in pred_nb_bins])

print("Naive Bayes — Evaluation:")
evaluate("Naive Bayes", y_test, pred_nb)

## 11. Model 3 — XGBoost
Gradient-boosted trees with regularisation and early stopping.

In [ ]:
xgb = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=30,
    eval_metric='rmse',
    verbosity=0,
    random_state=42
)
xgb.fit(
    X_train_sc, y_train,
    eval_set=[(X_test_sc, y_test)],
    verbose=False
)
pred_xgb = xgb.predict(X_test_sc)

print("XGBoost — Evaluation:")
evaluate("XGBoost", y_test, pred_xgb)

In [ ]:
# Feature importance plot
feat_imp = pd.Series(xgb.feature_importances_, index=X_train.columns)
top_20 = feat_imp.nlargest(20)

plt.figure(figsize=(10, 6))
top_20.sort_values().plot(kind='barh', color='steelblue', edgecolor='k', linewidth=0.5)
plt.title("XGBoost — Top 20 Feature Importances")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.savefig("xgboost_feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()

## 12. Model 4 — Stacked LSTM
Three-layer LSTM with Dropout for regularisation.  
Uses a 24-hour look-back window on scaled features.

In [ ]:
lstm_model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(TIME_STEPS, N_FEATURES)),
    Dropout(0.2),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dense(16, activation='relu'),
    Dense(1)
], name="Stacked_LSTM")

lstm_model.compile(optimizer=Adam(1e-3), loss='mse', metrics=['mae'])
lstm_model.summary()

In [ ]:
history_lstm = lstm_model.fit(
    X_tr_seq, y_tr_seq,
    epochs=100,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# Training curve
plt.figure(figsize=(10, 4))
plt.plot(history_lstm.history['loss'],     label='Train Loss')
plt.plot(history_lstm.history['val_loss'], label='Val Loss')
plt.title("LSTM — Training History")
plt.xlabel("Epoch"); plt.ylabel("MSE Loss")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("lstm_training_curve.png", dpi=150, bbox_inches='tight')
plt.show()

pred_lstm_sc = lstm_model.predict(X_te_seq, verbose=0).ravel()
pred_lstm = target_scaler.inverse_transform(pred_lstm_sc.reshape(-1, 1)).ravel()

print("\nLSTM — Evaluation:")
evaluate("LSTM", y_test_seq_orig, pred_lstm)

## 13. Model 5 — Transformer (Multi-Head Attention + FFN)

**Architecture:**
```
Input (seq_len, features)
  │
  ▼
Conv1D Projection → d_model dimensions
  │
  ▼  ┌─ repeated N times ─────────────────────────────────┐
     │  Multi-Head Self-Attention                          │
     │    └─ Add & LayerNorm  (residual connection)        │
     │  Feed-Forward Network: Dense(GELU) → Dense          │
     │    └─ Add & LayerNorm  (residual connection)        │
  ▼  └─────────────────────────────────────────────────────┘
Global Average Pooling  (sequence → vector)
  │
  ▼
MLP Head: Dense(64,relu) → Dense(32,relu) → Dense(1)
```

In [ ]:
def transformer_encoder_block(x, num_heads, key_dim, ff_dim, dropout_rate=0.1):
    """
    One Transformer encoder block:
      1. Multi-Head Self-Attention  → residual + LayerNorm
      2. Feed-Forward Network (GELU) → residual + LayerNorm
    """
    # Multi-Head Self-Attention
    attn_out = MultiHeadAttention(
        num_heads=num_heads, key_dim=key_dim, dropout=dropout_rate
    )(x, x)
    attn_out = Dropout(dropout_rate)(attn_out)
    x = LayerNormalization(epsilon=1e-6)(Add()([x, attn_out]))

    # Position-wise Feed-Forward Network
    ffn = Dense(ff_dim, activation='gelu')(x)   # GELU: smooth non-linearity
    ffn = Dropout(dropout_rate)(ffn)
    ffn = Dense(x.shape[-1])(ffn)               # project back to model dim
    ffn = Dropout(dropout_rate)(ffn)
    x = LayerNormalization(epsilon=1e-6)(Add()([x, ffn]))

    return x


def build_transformer(
    seq_len, n_features,
    d_model=64,
    num_heads=4,
    ff_dim=128,
    num_blocks=3,
    mlp_units=None,
    dropout=0.1
):
    if mlp_units is None:
        mlp_units = [64, 32]

    inp = Input(shape=(seq_len, n_features), name="sequence_input")

    # Linear projection: raw features → d_model
    x = Conv1D(filters=d_model, kernel_size=1, padding='same',
               name="input_projection")(inp)

    # Stacked Transformer encoder blocks
    for i in range(num_blocks):
        x = transformer_encoder_block(
            x,
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            ff_dim=ff_dim,
            dropout_rate=dropout
        )

    # Aggregate sequence → single vector
    x = GlobalAveragePooling1D(name="global_avg_pool")(x)

    # MLP prediction head
    for units in mlp_units:
        x = Dense(units, activation='relu')(x)
        x = Dropout(dropout)(x)

    output = Dense(1, name="forecast_output")(x)
    return Model(inputs=inp, outputs=output, name="Transformer_Forecaster")


transformer_model = build_transformer(
    seq_len=TIME_STEPS,
    n_features=N_FEATURES,
    d_model=64,
    num_heads=4,
    ff_dim=128,
    num_blocks=3,
    mlp_units=[64, 32],
    dropout=0.1
)

transformer_model.compile(optimizer=Adam(5e-4), loss='mse', metrics=['mae'])
transformer_model.summary()

In [ ]:
history_tf = transformer_model.fit(
    X_tr_seq, y_tr_seq,
    epochs=100,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# Training curve
plt.figure(figsize=(10, 4))
plt.plot(history_tf.history['loss'],     label='Train Loss')
plt.plot(history_tf.history['val_loss'], label='Val Loss')
plt.title("Transformer — Training History")
plt.xlabel("Epoch"); plt.ylabel("MSE Loss")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("transformer_training_curve.png", dpi=150, bbox_inches='tight')
plt.show()

pred_tf_sc = transformer_model.predict(X_te_seq, verbose=0).ravel()
pred_tf    = target_scaler.inverse_transform(pred_tf_sc.reshape(-1, 1)).ravel()

print("\nTransformer — Evaluation:")
evaluate("Transformer (MHA+FFN)", y_test_seq_orig, pred_tf)

## 14. Model 6 — ARIMA (5, 1, 0)
AutoRegressive Integrated Moving Average — classical univariate time-series model.

In [ ]:
arima_model = ARIMA(y_train, order=(5, 1, 0)).fit()
print(arima_model.summary())

In [ ]:
pred_arima = arima_model.forecast(steps=len(y_test))

print("ARIMA — Evaluation:")
evaluate("ARIMA", y_test, pred_arima)

## 15. Model 7 — SARIMA (1,1,1)(1,1,1,24)
Seasonal ARIMA with 24-hour seasonal period (captures daily solar cycle).

In [ ]:
sarima_model = SARIMAX(
    y_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print(sarima_model.summary())

In [ ]:
pred_sarima = sarima_model.forecast(steps=len(y_test))

print("SARIMA — Evaluation:")
evaluate("SARIMA", y_test, pred_sarima)

## 16. Final Results Comparison

In [ ]:
results_df = pd.DataFrame(
    results, columns=["Model", "MAE (kW)", "RMSE (kW)", "R² Score"]
).sort_values("R² Score", ascending=False).reset_index(drop=True)

results_df.index += 1  # rank from 1

print("=" * 65)
print(results_df.to_string())
print("=" * 65)

best = results_df.iloc[0]
print(f"\n🏆  BEST MODEL : {best['Model']}")
print(f"    MAE        : {best['MAE (kW)']} kW")
print(f"    RMSE       : {best['RMSE (kW)']} kW")
print(f"    R² Score   : {best['R² Score']}")

## 17. Forecast Visualisations

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 14))
fig.suptitle("Solar PV AC Power — Forecast Comparison", fontsize=16, fontweight='bold')

# Plot 1: Deep Learning models
ax = axes[0]
ax.plot(y_test_seq_orig[-200:], label='Actual',              color='black',    linewidth=1.5)
ax.plot(pred_lstm[-200:],       label='LSTM',                color='royalblue', alpha=0.85)
ax.plot(pred_tf[-200:],         label='Transformer (MHA+FFN)', color='crimson',  alpha=0.85)
ax.set_title("Deep Learning Models — Last 200 Test Hours")
ax.set_ylabel("AC Power (kW)"); ax.legend(); ax.grid(alpha=0.3)

# Plot 2: Classical models
ax = axes[1]
ax.plot(y_test.values[-200:],  label='Actual',  color='black',       linewidth=1.5)
ax.plot(pred_xgb[-200:],       label='XGBoost', color='seagreen',    alpha=0.85)
ax.plot(pred_arima[-200:],     label='ARIMA',   color='darkorange',  alpha=0.85)
ax.plot(pred_sarima[-200:],    label='SARIMA',  color='purple',      alpha=0.85)
ax.set_title("Classical & Ensemble Models — Last 200 Test Hours")
ax.set_ylabel("AC Power (kW)"); ax.legend(); ax.grid(alpha=0.3)

# Plot 3: R² bar chart
ax = axes[2]
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(results_df))]
bars = ax.barh(results_df["Model"][::-1], results_df["R² Score"][::-1],
               color=colors[::-1], edgecolor='k', linewidth=0.6)
ax.set_xlabel("R² Score (higher is better)")
ax.set_title("Model Comparison — R² Score")
ax.axvline(0, color='k', linewidth=0.8)
for bar, val in zip(bars, results_df["R² Score"][::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ All charts saved.")

In [ ]:
# Scatter plot: Actual vs Predicted (best deep-learning model)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (pred, name, color) in zip(axes, [
    (pred_lstm, "LSTM",                "royalblue"),
    (pred_tf,   "Transformer (MHA+FFN)", "crimson")
]):
    ax.scatter(y_test_seq_orig, pred, alpha=0.3, s=5, color=color)
    lim = [min(y_test_seq_orig.min(), pred.min()),
           max(y_test_seq_orig.max(), pred.max())]
    ax.plot(lim, lim, 'k--', linewidth=1)
    ax.set_title(f"{name} — Actual vs Predicted")
    ax.set_xlabel("Actual AC Power (kW)")
    ax.set_ylabel("Predicted AC Power (kW)")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("scatter_actual_vs_predicted.png", dpi=150, bbox_inches='tight')
plt.show()